In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/jakenty1/vnmathproblems2026/problems.csv
/kaggle/input/datasets/jakenty1/vnmathproblems2026/problem.math.csv


In [2]:
model_choice = "nemotron-3-nano:4b"# "deepseek-coder:6.7b-instruct"# "llama3.1:8b"#"gemma2:9b"#"qwen2.5:7b" #"gemma2:9b"#"mistral:7b-instruct"  # Change as needed, e.g. "gemma2:9b", "qwen2.5:7b", "llama3.1:8b"
start_problem = 1
end_problem = 50
system_prompt = "Output only the final result, no explanations."

In [3]:
!head /kaggle/input/datasets/jakenty1/vnmathproblems2026/problem.math.csv -n 1


# LLM Math Reasoning Experiment on Kaggle

This notebook runs math-reasoning prompt experiments using Ollama on Kaggle GPU.

**Kaggle Setup:**
1. **Settings**: Enable **Internet** to download Ollama and models.
2. **Accelerator**: Select **GPU T4 x2** or **GPU P100**.
3. **Input**: Add the dataset containing `problem.math.csv`, for example `jakenty1/vnmathproblems2026`.

The notebook writes result files with `.math` in the output filename.


In [4]:
!pip install zstandard zstd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 55.3 MB/s eta 0:00:00


In [5]:
!sudo apt-get install zstd




The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 133 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (8,591 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
Selecting previously unselected package zstd.
(Reading database ... 124626 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [6]:
# Install Ollama
import os
import subprocess
import time
import requests
import json
import re
import csv
from datetime import datetime

print("Installing Ollama via official script...")
# The official script is often more reliable than direct binary curl in some environments
!curl -fsSL https://ollama.com/install.sh | sh



Installing Ollama via official script...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [7]:
print("Starting Ollama server...")
# Set environment variables for Kaggle environment
os.environ["OLLAMA_HOST"] = "0.0.0.0"
os.environ["OLLAMA_ORIGINS"] = "*"

# Start Ollama server in background
subprocess.Popen(["ollama", "serve"],
                 stdout=open("ollama_stdout.log", "w"),
                 stderr=open("ollama_stderr.log", "w"))

# Wait for server to be ready
print("Waiting for Ollama server to respond...")
for i in range(30):
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=2)
        if r.status_code == 200:
            print(f"Ollama server ready after {i+1}s")
            break
    except:
        pass
    time.sleep(1)
else:
    print("ERROR: Ollama server failed to start.")
    print("Last 20 lines of ollama_stderr.log:")
    !tail -n 20 ollama_stderr.log

Starting Ollama server...
Waiting for Ollama server to respond...
Ollama server ready after 2s


In [8]:
!ollama list

]11;?\NAME    ID    SIZE    MODIFIED 


## 2. Math Configuration and Data Loading


In [9]:
def call_ollama(model, system_prompt, user_prompt):
    start = time.time()
    try:
        payload = {
            "model": model,
            "prompt": user_prompt,
            "stream": False,
            "options": {"temperature": 0.2, "num_predict": 8192},
        }
        if system_prompt:
            payload["system"] = system_prompt
        r = requests.post("http://localhost:11434/api/generate", json=payload, timeout=300)
        r.raise_for_status()
        data = r.json()
        return data.get("response", ""), {
            "duration": time.time() - start,
            "prompt_tokens": data.get("prompt_eval_count", 0),
            "completion_tokens": data.get("eval_count", 0),
            "success": True,
            "error": "",
        }
    except Exception as e:
        return "", {
            "duration": time.time() - start,
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "success": False,
            "error": str(e),
        }


import ast
import math
import operator


def normalize_response_text(response: str) -> str:
    """Keep only the likely final answer from a model response."""
    text = str(response).strip()
    text = re.sub(r"```(?:text|json|python)?\s*", "", text, flags=re.IGNORECASE)
    text = text.replace("```", "").strip()

    marker_patterns = [
        r"(?:final answer|answer|result)\s*(?:is|:)?\s*(.+)$",
        r"(?:therefore|so),?\s*(.+)$",
    ]
    for pattern in marker_patterns:
        matches = re.findall(pattern, text, flags=re.IGNORECASE | re.DOTALL)
        if matches:
            text = matches[-1].strip()
            break

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if lines:
        text = lines[-1]
    return text.strip().strip(".")


def _replace_latex_frac(text: str) -> str:
    pattern = re.compile(r"\\frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}")
    previous = None
    while previous != text:
        previous = text
        text = pattern.sub(r"(\1)/(\2)", text)
    return text


def normalize_math_expression(value) -> str:
    text = str(value).strip()
    text = text.strip("$ ")
    text = text.replace("\\(", "").replace("\\)", "")
    text = text.replace("\\[", "").replace("\\]", "")
    text = text.replace("{,}", ",")
    text = text.replace(",", "")
    text = text.replace("−", "-").replace("–", "-").replace("—", "-")
    text = text.replace("×", "*").replace("·", "*").replace("\\cdot", "*")
    text = text.replace("^", "**")
    text = _replace_latex_frac(text)
    text = re.sub(r"\\sqrt\s*\{([^{}]+)\}", r"sqrt(\1)", text)
    text = re.sub(r"√\s*([0-9.]+)", r"sqrt(\1)", text)
    return text.strip()


_ALLOWED_BINARY = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
}
_ALLOWED_UNARY = {ast.UAdd: operator.pos, ast.USub: operator.neg}
_ALLOWED_NAMES = {"pi": math.pi, "e": math.e}
_ALLOWED_FUNCS = {"sqrt": math.sqrt}


def _safe_eval(node):
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return float(node.value)
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_BINARY:
        return _ALLOWED_BINARY[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_UNARY:
        return _ALLOWED_UNARY[type(node.op)](_safe_eval(node.operand))
    if isinstance(node, ast.Name) and node.id in _ALLOWED_NAMES:
        return _ALLOWED_NAMES[node.id]
    if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in _ALLOWED_FUNCS and len(node.args) == 1:
        return _ALLOWED_FUNCS[node.func.id](_safe_eval(node.args[0]))
    raise ValueError(f"Unsupported math expression: {ast.dump(node)}")


def parse_math_value(value):
    """Parse decimals, fractions, percentages, and basic LaTeX/math formulas."""
    if isinstance(value, (int, float)):
        return float(value)
    text = normalize_math_expression(value)
    if not text:
        raise ValueError("empty math value")
    is_percent = text.endswith("%")
    if is_percent:
        text = text[:-1].strip()
    parsed = ast.parse(text, mode="eval")
    result = float(_safe_eval(parsed))
    return result / 100 if is_percent else result


def extract_numbers(text: str):
    """Extract numeric values from text, including fractions and simple LaTeX formulas."""
    text = normalize_response_text(text)

    candidates = [text]
    if "=" in text:
        candidates.append(text.split("=")[-1].strip())
    for candidate in candidates:
        try:
            return [parse_math_value(candidate)]
        except Exception:
            pass

    values = []
    token_pattern = re.compile(
        r"\\frac\s*\{[^{}]+\}\s*\{[^{}]+\}"
        r"|\\sqrt\s*\{[^{}]+\}"
        r"|sqrt\s*\([^()]+\)"
        r"|√\s*[0-9.]+"
        r"|[-+]?\d[\d,]*(?:\.\d+)?\s*/\s*[-+]?\d[\d,]*(?:\.\d+)?"
        r"|[-+]?\d[\d,]*(?:\.\d+)?\s*%?"
        r"|[-+]?\.\d+\s*%?"
    )
    for match in token_pattern.finditer(text):
        try:
            values.append(parse_math_value(match.group(0)))
        except Exception:
            continue
    return values


def parse_expected_answer(raw_value: str):
    """Parse expected_answer from problem.math.csv."""
    try:
        loaded = json.loads(raw_value)
    except Exception:
        loaded = raw_value

    if isinstance(loaded, list):
        return [parse_math_value(item) for item in loaded]
    return parse_math_value(loaded)


def almost_equal(actual, expected, tolerance=1e-6):
    try:
        return abs(parse_math_value(actual) - parse_math_value(expected)) <= tolerance
    except Exception:
        return False


def evaluate_math_answer(response_text: str, expected_answer, tolerance=1e-6):
    numbers = extract_numbers(response_text)
    expected_values = expected_answer if isinstance(expected_answer, list) else [expected_answer]

    errors = []
    if len(numbers) < len(expected_values):
        errors.append(
            f"Expected {len(expected_values)} numeric value(s), extracted {len(numbers)}: {numbers}"
        )
        return {
            "passed": False,
            "values_passed": 0,
            "values_total": len(expected_values),
            "extracted_values": numbers,
            "errors": errors,
        }

    candidate_values = numbers[-len(expected_values):]
    passed_count = 0
    for actual, expected in zip(candidate_values, expected_values):
        if almost_equal(actual, expected, tolerance=tolerance):
            passed_count += 1
        else:
            errors.append(f"Expected {expected}, got {actual}")

    return {
        "passed": passed_count == len(expected_values),
        "values_passed": passed_count,
        "values_total": len(expected_values),
        "extracted_values": candidate_values,
        "errors": errors,
    }

def parse_csv_row(row):
    return {
        "task_id": f"{row['task_id']}_{row['category_id']}",
        "base_task_id": row["task_id"],
        "category_id": row["category_id"],
        "description": row["prompt"].strip(),
        "expected_answer": parse_expected_answer(row["expected_answer"]),
        "difficulty": row.get("difficulty", ""),
        "source_task_id": row.get("source_task_id", ""),
        "source_old_id": row.get("source_old_id", ""),
    }


def get_completed_ids(filename):
    if not os.path.exists(filename):
        return set()
    completed = set()
    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            try:
                data = json.loads(line)
                if str(data.get("response_text", "")).strip():
                    completed.add(data.get("task_id"))
            except Exception:
                continue
    return completed


def append_result(filename, result):
    with open(filename, "a", encoding="utf-8") as f:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")


csv_path = "/kaggle/input/datasets/jakenty1/vnmathproblems2026/problem.math.csv"
if not os.path.exists(csv_path):
    csv_path = "/kaggle/input/vnmathproblems2026/problem.math.csv"
if not os.path.exists(csv_path):
    csv_path = "/kaggle/working/problem.math.csv"

print(f"Using math problem CSV: {csv_path}")


Using math problem CSV: /kaggle/input/datasets/jakenty1/vnmathproblems2026/problem.math.csv


In [10]:


print(f"Pulling model {model_choice}...")
!ollama pull {model_choice}

output_filename = f"/kaggle/working/{model_choice.replace(':', '-')}_{start_problem}_{end_problem}.math.jsonl"

# Seed Kaggle working output from the uploaded previous-result dataset.
# Empty response rows are intentionally skipped so those tasks rerun.
seed_basename = f"{model_choice.replace(':', '-')}_{start_problem}_{end_problem}.math.jsonl"
seed_candidates = [
    f"/kaggle/input/nemotronresultv1/{seed_basename}",
    f"/kaggle/input/datasets/jakenty1/nemotronresultv1/{seed_basename}",
]
seed_filename = next((p for p in seed_candidates if os.path.exists(p)), None)
print("Resume seed candidates:", seed_candidates)
print("Selected seed file:", seed_filename or "NOT FOUND")

if seed_filename:
    rows_by_task_id = {}
    existing_kept_rows = 0
    existing_empty_rows = 0
    seed_added_rows = 0
    seed_duplicate_rows = 0
    seed_empty_rows = 0

    if os.path.exists(output_filename):
        with open(output_filename, "r", encoding="utf-8") as existing:
            for line in existing:
                if not line.strip():
                    continue
                try:
                    data = json.loads(line)
                except Exception:
                    continue
                task_id = data.get("task_id")
                if str(data.get("response_text", "")).strip() and task_id:
                    rows_by_task_id[task_id] = data
                    existing_kept_rows += 1
                else:
                    existing_empty_rows += 1

    with open(seed_filename, "r", encoding="utf-8") as seed:
        for line in seed:
            if not line.strip():
                continue
            try:
                data = json.loads(line)
            except Exception:
                continue
            task_id = data.get("task_id")
            if not str(data.get("response_text", "")).strip():
                seed_empty_rows += 1
                continue
            if task_id in rows_by_task_id:
                seed_duplicate_rows += 1
                continue
            rows_by_task_id[task_id] = data
            seed_added_rows += 1

    with open(output_filename, "w", encoding="utf-8") as dst:
        for data in rows_by_task_id.values():
            dst.write(json.dumps(data, ensure_ascii=False) + "\n")

    print(
        f"Prepared resume file {output_filename}: "
        f"completed={len(rows_by_task_id)}, existing_kept={existing_kept_rows}, "
        f"existing_empty_removed={existing_empty_rows}, seed_added={seed_added_rows}, "
        f"seed_duplicate={seed_duplicate_rows}, seed_empty_skipped={seed_empty_rows}."
    )
else:
    try:
        print("Available /kaggle/input directories:", os.listdir("/kaggle/input"))
    except Exception as e:
        print("Could not list /kaggle/input:", e)

completed_ids = get_completed_ids(output_filename)

problems_to_test = []
with open(csv_path, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        try:
            numeric_source_id = int(row.get("source_task_id") or row["task_id"].replace("math_", ""))
            if start_problem <= numeric_source_id <= end_problem:
                prob = parse_csv_row(row)
                if prob["task_id"] not in completed_ids:
                    problems_to_test.append(prob)
        except Exception:
            continue

print(f"Loaded {len(problems_to_test)} remaining math tasks for range {start_problem}-{end_problem}.")

for prob in problems_to_test:
    print(f"Testing {prob['task_id']}...", end=" ", flush=True)
    response_text, meta = call_ollama(model_choice, system_prompt, prob["description"])
    if not meta["success"]:
        print(f"Error: {meta['error']}")
        continue

    eval_res = evaluate_math_answer(response_text, prob["expected_answer"])
    print(f"{'PASS' if eval_res['passed'] else 'FAIL'} ({eval_res['values_passed']}/{eval_res['values_total']})")

    result = {
        "timestamp": datetime.now().isoformat(),
        "model": model_choice,
        "task_id": prob["task_id"],
        "base_task_id": prob["base_task_id"],
        "category_id": prob["category_id"],
        "difficulty": prob["difficulty"],
        "source_task_id": prob["source_task_id"],
        "source_old_id": prob["source_old_id"],
        "passed": eval_res["passed"],
        "values_passed": eval_res["values_passed"],
        "values_total": eval_res["values_total"],
        "expected_answer": prob["expected_answer"],
        "extracted_values": eval_res["extracted_values"],
        "errors": eval_res["errors"],
        "duration_seconds": meta["duration"],
        "prompt_tokens": meta["prompt_tokens"],
        "completion_tokens": meta["completion_tokens"],
        "response_text": response_text,
    }
    append_result(output_filename, result)

print(f"Results saved to {output_filename}")


Pulling model nemotron-3-nano:4b...
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling 527db2cf6c70:   0% ▕                  ▏  13 MB/2.8 GB                  pulling manifest 
pulling 527db2cf6c70:   2% ▕                  ▏  50 MB/2.8 GB                  pulling manifest 
pulling 527db2cf6c70:   5% ▕                  ▏ 133 MB/2.8 GB                  pulling manifest 
pulling 527db2cf6c70:   8% ▕█                 ▏ 224 MB/2.8 GB                  pulling manifest 
pulling 527db2cf6c70:   9% ▕█                 ▏ 268 MB/2.8 GB                  pulling manifest 
pulling 527db2cf6c70:  13% ▕██                ▏ 369 MB/2.8 GB                  pulling manifest 
pulling 527db2cf6c70:  16% ▕██                ▏ 461 MB/2.8 GB                  pulling manifest 
pulling 527db2cf6c70:  18% ▕███               ▏ 509 MB/2.8 GB        

## 4. Run Additional Ranges
Edit `start_problem`, `end_problem`, and `model_choice` above, then rerun the final cell to process new math tasks without restarting. Output files use `.math.jsonl`.
